# BLOQUE 02 – Análisis de movilidad
## Objetivo
Integrar las capas de la previa etapa en un solo geopackage, y realizar el filtrado de lo que se necesitará en la etapa 3.
## Inputs
- 1_D2C1_Poblacion.gpkg
- 1_D2C2_Económico.gpkg
## Procesos
- Integración
- Filtrado
## Output
- 2_Dimensiones.gpkg
    - 2_D4_Movilidad.shp
      

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [2]:
ruta_movilidad = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_1\1_D4C1_Movilidad.gpkg"

ruta_salida = r"C:\Users\alflo\Desktop\DAL_LAB\outputs\etapa_2\2_Dimensiones.gpkg"

# Carga de datos

In [3]:
gpd.list_layers(ruta_movilidad)

,name,geometry_type
0,1_D4C1V1_ciclovias,MultiLineString
1,1_D4C1V1_estaciones_mibici,Point
2,1_D4C1V2_transporte_público,MultiLineString
3,1_D4C1V2_transporte_público_paradas,MultiPoint
4,1_D4C1V3_estructura_vial,MultiLineString


In [4]:
cv = gpd.read_file(ruta_movilidad, layer = "1_D4C1V1_ciclovias")
mibici = gpd.read_file(ruta_movilidad, layer = "1_D4C1V1_estaciones_mibici")
tp = gpd.read_file(ruta_movilidad, layer = "1_D4C1V2_transporte_público")
tp_p = gpd.read_file(ruta_movilidad, layer = "1_D4C1V2_transporte_público_paradas")
vial = gpd.read_file(ruta_movilidad, layer = "1_D4C1V3_estructura_vial")

In [5]:
cv.crs

<Projected CRS: EPSG:6372>
Name: Mexico ITRF2008 / LCC
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: Mexico - onshore and offshore.
- bounds: (-122.19, 12.1, -84.64, 32.72)
Coordinate Operation:
- name: Mexico LCC
- method: Lambert Conic Conformal (2SP)
Datum: Mexico ITRF2008
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [5]:
cv.columns

Index(['nombre_de_', 'municipio', 'Categoría', 'geometry'], dtype='object')

In [6]:
mibici.columns

Index(['clave_de_i', 'Categoría', 'geometry'], dtype='object')

In [7]:
tp.columns

Index(['Tipo', 'Categoría', 'geometry'], dtype='object')

In [8]:
tp_p.columns

Index(['Linea', 'Categoría', 'Tipo', 'Estructura', 'geometry'], dtype='object')

In [9]:
vial.columns

Index(['Categoría', 'geometry'], dtype='object')

In [6]:
cv['Tipo'] = 'Ciclovía'
tp['Tipo'] = 'Trans_Publico'
vial['Tipo'] = 'Vialidad'

mibici['Tipo'] = 'Estación Bici'
mibici = mibici[['Tipo','Categoría', 'geometry']]

# Unión a un solo df, extracción de columnas de interés

In [7]:
lineas = pd.concat([cv[['Tipo','Categoría', 'geometry']], tp[['Tipo','Categoría', 'geometry']], vial[['Tipo','Categoría', 'geometry']]])
puntos = pd.concat([tp_p[['Tipo','Categoría', 'geometry']], mibici[['Tipo','Categoría', 'geometry']]])

# filtrado estado y municipios

In [12]:
# # EXTRAER JALISCO
# ue_ent = ue[ue['cve_ent'] == '14'].copy()
# eq_ent = eq[eq['CVE_ENT'] == '14'].copy()

# # Lista de municipios AMG
# municipios = ['002','039', '044', '051','070','097', '098', '101', '120', '124']

# # Filtrar el DataFrame solo con los municipios de interés
# ue_mun = ue_ent[
#     (ue_ent['cve_mun'].isin(municipios)) & 
#     (ue_ent['cve_ent'] == '14')
# ]

# eq_mun = eq_ent[
#     (eq_ent['CVE_MUN'].isin(municipios)) & 
#     (eq_ent['CVE_ENT'] == '14')
# ]

In [13]:
# # Reemplazar * con 1
# def replace(dataframe):
#     return dataframe.replace('*', '1')

# # Limpiar *
# ue_mun = replace(ue_mun)
# eq_mun = replace(eq_mun)

# # Reemplazar N/D
# ue_mun = ue_mun.replace('N/D', np.nan)
# eq_mun = eq_mun.replace('N/D', np.nan)

# # Actualizar valores sin información de "null" a 0
# ue_mun = ue_mun.fillna(0)
# eq_mun = eq_mun.fillna(0)

# Guardado 

In [9]:
puntos.crs

<Projected CRS: EPSG:6372>
Name: Mexico ITRF2008 / LCC
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: Mexico - onshore and offshore.
- bounds: (-122.19, 12.1, -84.64, 32.72)
Coordinate Operation:
- name: Mexico LCC
- method: Lambert Conic Conformal (2SP)
Datum: Mexico ITRF2008
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [11]:
lineas.to_file(ruta_salida, layer='2_D4_movilidad_lineas', driver='GPKG')
puntos.to_file(ruta_salida, layer='2_D4_movilidad_puntos', driver='GPKG')